In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
df = pd.read_parquet("train.parquet")

In [5]:
df.describe()

,seq_ix,step_in_seq,need_prediction,0,1,2,3,4,5,6,...,22,23,24,25,26,27,28,29,30,31
count,517000.00000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,...,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000,517000.000000
mean,258.00000,499.500000,0.899000,0.001247,0.022680,-0.006079,-0.013578,-0.001519,0.006435,-0.022507,...,0.021797,-0.015975,-0.010245,0.003196,-0.022505,-0.021408,0.021334,-0.007824,0.006495,-0.018510
std,149.24491,288.675269,0.301329,0.985162,0.994129,1.008178,1.010985,0.990943,0.989856,1.001293,...,0.989687,0.989816,0.991879,0.997269,0.997177,0.996491,0.995721,0.996676,1.002600,1.001944
min,0.00000,0.000000,0.000000,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,...,-5.199338,-3.400935,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338,-5.199338
25%,129.00000,249.750000,1.000000,-0.672111,-0.655956,-0.680784,-0.703100,-0.684677,-0.668105,-0.709381,...,-0.666198,-0.696765,-0.671657,-0.679927,-0.688384,-0.697000,-0.655391,-0.694185,-0.658455,-0.684046
50%,258.00000,499.500000,1.000000,-0.003134,0.013809,-0.006287,-0.020027,-0.003576,0.009730,-0.018166,...,0.030758,-0.010643,-0.006489,0.011550,-0.013160,-0.004827,0.017762,-0.012549,0.008081,-0.012764
75%,387.00000,749.250000,1.000000,0.655925,0.698124,0.663186,0.661700,0.670718,0.685277,0.669402,...,0.687280,0.663009,0.650527,0.684317,0.651541,0.652707,0.700777,0.669457,0.680425,0.643227
max,516.00000,999.000000,1.000000,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,...,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338,5.199338


In [6]:
seqs = np.zeros((517,1000,33))
for i in range(517):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["seq_ix","step_in_seq"]).values


seqs = np.diff(seqs,n=1,axis =1)
seqs.shape
# seqs[i][j][0] gives ith sequence jth row need_prediction


(517, 999, 33)

In [7]:
input_to_encoder_seqs = np.zeros((517,100,32))
for i in range(517):
   input_to_encoder_seqs[i] = df[(df["seq_ix"] == i) & (df["need_prediction"]==0)].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).iloc[:-1].values

In [8]:
output_of_decoder_seqs = np.zeros((517,899,32))
for i in range(517):
   output_of_decoder_seqs[i] = df[(df["seq_ix"] == i) & (df["need_prediction"]==1)].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values

In [9]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [10]:
class Preprocessor:
    def __init__(self):
        self.scaler = MinMaxScaler()
    def fit(self,X_train):
        X = X_train.reshape(-1,X_train.shape[-1])
        #print(X.shape)
        X = self.scaler.fit_transform(X).reshape(X_train.shape)
        return X
    def transform(self,X_test):
        X = X_test.reshape(-1,X_test.shape[-1])
        return self.scaler.transform(X).reshape(X_test.shape)

In [11]:
preproc = Preprocessor()
X_train_temp ,X_test_temp, y_train , y_test = train_test_split(input_to_encoder_seqs,output_of_decoder_seqs,random_state=SEED,train_size=0.8)
X_train = preproc.fit(X_train_temp)
X_test = preproc.transform(X_test_temp)

In [ ]:
class AutoEncoder(Model):
  def __init__(self,input_shape):
    self.num_neurons = [40,40,40]
    
    self.enc_dims = 40
    self.decoder_neurons = self.num_neurons[-1]
    self.to_predict = 899
    # self.encoder_layers = []
    # self.decoder_layers = []
    super(AutoEncoder,self).__init__()
    
    
    self.encoder_lstm_1 = layers.LSTM(self.num_neurons[0],
                                          return_sequences=True,
                                          name="encoder_lstm_1") 
    self.encoder_lstm_middle = layers.LSTM(self.num_neurons[1],
                                          return_sequences=True,
                                          name="encoder_lstm_middle")
    self.projector_layer = layers.Dense(self.num_neurons[1],activation="linear",name = "projector")
    
    self.encoder_lstm_2 = layers.LSTM(self.num_neurons[2],
                                          return_sequences=True,
                                          return_state=True,
                                          name="encoder_lstm_2")

    self.decoder_lstm = layers.LSTM(self.decoder_neurons,
                                         return_sequences=True,
                                         return_state=True,
                                         name="decoder_lstm")
    


    self.DenseQ = layers.Dense(self.enc_dims,name = "Query_Dense")
    self.DenseK = layers.Dense(self.enc_dims,name = "Query_Key")
    self.DenseV = layers.Dense(self.enc_dims,name = "Query_Value")

    num_heads = 4
    self.self_attention_layer = layers.MultiHeadAttention(name = "self_attention",num_heads = num_heads,key_dim = self.enc_dims//num_heads)
    self.cross_attention_layer = layers.MultiHeadAttention(name="attention",num_heads = num_heads,key_dim= self.enc_dims//num_heads)
    self.concat_layer = layers.Concatenate(axis=-1, name="concat_attention")
    self.output_dense_layer = layers.Dense(input_shape[1], name="output_dense",activation="linear") 
    self.output_time_dist = layers.TimeDistributed(self.output_dense_layer, name="output_time_distributed") 



    inputs = tf.keras.Input(input_shape,name = "init_input")
    #mean_inputs = layers.Lambda(lambda x: tf.math.reduce_mean(x,axis=1,keepdims=True))(inputs)
    x1 = self.encoder_lstm_1(inputs)
    x = self.encoder_lstm_middle(x1)
    x1_proj = self.projector_layer(x1)
    x = layers.Add(name = "skip_conn")([x1_proj,x])
    lstm_out , h , c = self.encoder_lstm_2(x)
    states = [h,c]

    key = self.DenseK(lstm_out)
    query = self.DenseQ(lstm_out)
    val = self.DenseV(lstm_out)

    attention_context_for_enc = self.self_attention_layer(query, val,key)
    encoder_out = self.concat_layer([attention_context_for_enc, lstm_out])


    self.encoder_model = Model(inputs = inputs,outputs = [encoder_out] + states , name = "ENCODER")

    target = tf.keras.Input(shape = (self.to_predict,input_shape[1]))
    decoder_lstm_outputs, _, _ = self.decoder_lstm(target,initial_state=states)

    attention_context = self.cross_attention_layer(decoder_lstm_outputs, encoder_out,encoder_out)
    decoder_combined_context = self.concat_layer([attention_context, decoder_lstm_outputs])
    decoder_pred_outputs = self.output_time_dist(decoder_combined_context)


    self.training_model = Model(inputs=[inputs, target],outputs=decoder_pred_outputs)

    #----------------------------------------------------------------------------------------------------
    inf_decoder_input_prev = Input(shape=(1, input_shape[1]), name="inf_decoder_input") # Previous prediction
    inf_decoder_state_h = Input(shape=(self.decoder_neurons,), name="inf_state_h")          # Previous h state
    inf_decoder_state_c = Input(shape=(self.decoder_neurons,), name="inf_state_c")          # Previous c state
    inf_decoder_states_prev = [inf_decoder_state_h, inf_decoder_state_c]                 # Combined previous states
    inf_encoder_outputs = Input(shape=(input_shape[0], self.num_neurons[-1]+self.enc_dims), name="inf_encoder_outputs") # ALL encoder outputs for attention

        # Connect layers for one step using shared layer instances:
    inf_dec_lstm_out, inf_state_h_out, inf_state_c_out = self.decoder_lstm(
        inf_decoder_input_prev, initial_state=inf_decoder_states_prev # Initialize with previous states
    )
    inf_attention_context = self.cross_attention_layer(inf_dec_lstm_out, inf_encoder_outputs,inf_encoder_outputs) # Calculate attention for this step
    inf_combined_context = self.concat_layer([inf_attention_context, inf_dec_lstm_out])    # Combine
    inf_decoder_pred = self.output_time_dist(inf_combined_context)                       # Predict (shape=(batch, 1, features))

        # Define the single-step inference decoder model
    self.inf_decoder_model = Model(
        inputs=[inf_decoder_input_prev, inf_decoder_states_prev, inf_encoder_outputs],
        outputs=[inf_decoder_pred] + [inf_state_h_out, inf_state_c_out], # Output prediction AND next states
        name="inf_decoder"
    )      

    
                            
  

  def train(self,X_train,y_train):
    last_known_step = X_train[:, -1:, :] 
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_r2_keras',  
        factor=0.5,          
        patience=2,          
        verbose=1,           
        mode='max',         
        min_delta=0.002,      
        cooldown=2,          
        min_lr=0.0002        
    )
    # 2. GET THE FUTURE SEQUENCE, *EXCEPT* ITS LAST STEP
    # Shape: (batch_size, 898, 32)
    future_inputs = y_train[:, :-1, :]
    
    # 3. CREATE THE NEW DECODER INPUT VARIABLE
    # This is the "shifted" sequence for teacher forcing
    # Shape: (batch_size, 899, 32)
    decoder_input_data = np.concatenate([last_known_step, future_inputs], axis=1)
    
    # 4. THE TARGET IS JUST THE ORIGINAL Y_TRAIN
    decoder_target_data = y_train
    optim = tf.keras.optimizers.Adam(learning_rate=0.01,clipnorm = 1.0)
    self.training_model.compile(optimizer = optim,loss= "mse",metrics=[r2_keras])
    self.training_model.summary()
    self.training_model.fit([X_train,decoder_input_data],
                            y_train,batch_size = 32, 
                            epochs = 30,
                            verbose = True, 
                            validation_split = 0.2,
                            callbacks = [reduce_lr])
    self.training_model.save_weights("C:/CodingWorkspace/Python/WunderRNN/AE_wts.weights.h5")
    


In [15]:
autoenc = AutoEncoder([100,32])
autoenc.train(X_train = input_to_encoder_seqs,y_train=output_of_decoder_seqs)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ init_input          │ (None, 100, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ (None, 100, 40)   │     11,680 │ init_input[0][0]  │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projector (Dense)   │ (None, 100, 40)   │      1,640 │ encoder_lstm_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_middle │ (None, 100, 40)   │     12,960 │ encoder_lstm_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_conn (Add)     │ (None, 100, 40)   │          0 │ projector[0][0],  │
│                     │                   │            │ encoder_lstm_mid… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 100, 40), │     12,960 │ skip_conn[0][0]   │
│ (LSTM)              │ (None, 40),       │            │                   │
│                     │ (None, 40)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Dense (Dense) │ (None, 100, 40)   │      1,640 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Value (Dense) │ (None, 100, 40)   │      1,640 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Key (Dense)   │ (None, 100, 40)   │      1,640 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 899, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 100, 40)   │      6,560 │ Query_Dense[0][0… │
│ (MultiHeadAttentio… │                   │            │ Query_Value[0][0… │
│                     │                   │            │ Query_Key[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, 1, 80)     │          0 │ self_attention[0… │
│ (Concatenate)       │                   │            │ encoder_lstm_2[0… │
│                     │                   │            │ attention[0][0],  │
│                     │                   │            │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 40),   │     11,680 │ input_layer_1[0]… │
│                     │ (None, 40),       │            │ encoder_lstm_2[0… │
│                     │ (None, 40)]       │            │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 1, 40)     │      9,760 │ decoder_lstm[0][… │
│ (MultiHeadAttentio… │                   │            │ concat_attention… │
│                     │                   │            │ concat_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_time_distri… │ (None, 1, 32)     │      2,592 │ concat_attention… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 74,752 (292.00 KB)

 Trainable params: 74,752 (292.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - loss: 0.8053 - r2_keras: 0.1835 - val_loss: 0.7605 - val_r2_keras: 0.2262 - learning_rate: 0.0100
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 821ms/step - loss: 0.7088 - r2_keras: 0.2802 - val_loss: 0.7298 - val_r2_keras: 0.2588 - learning_rate: 0.0100
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 686ms/step - loss: 0.6775 - r2_keras: 0.3119 - val_loss: 0.7124 - val_r2_keras: 0.2789 - learning_rate: 0.0100
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 678ms/step - loss: 0.6599 - r2_keras: 0.3298 - val_loss: 0.7015 - val_r2_keras: 0.2922 - learning_rate: 0.0100
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 687ms/step - loss: 0.6500 - r2_keras: 0.3398 - val_loss: 0.6939 - val_r2_keras: 0.3011 - learning_rate: 0.0100
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 662ms/step - loss: 0.6425 - r2_keras: 0.3475 - val_loss: 0.6884 - val_r2_keras: 0.3077 - learning_rate: 0.0100
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 666ms/step - loss: 0.6366 - r2_keras: 0.3534 - v